# 📘 Notebook 02: Regular Expressions

### Course: *From Words to Meaning: Natural Language Processing in Python*
**Instructor:** Ioannis Tsioulis

*Module A: Text as Data · Notebook 2 of 3 in this module · (2 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain what a **regular expression** is and when it is the right tool
- Write patterns with **character classes**, **quantifiers**, **anchors** and **groups**
- Use `re.search`, `re.findall`, `re.sub` and `re.split` with confidence
- Tell **greedy** matching from **lazy** matching, and choose between them
- Build a **text-cleaning pipeline** for messy real-world text
- Write a tokeniser as a single pattern, and say what regular expressions **cannot** do

> **Prerequisites:** Notebook 01.
>
> 🔭 **Why this notebook matters:** real text is dirty. It arrives with web addresses, e-mail addresses, HTML tags, stray symbols and inconsistent spacing. Regular expressions are the standard tool for cleaning it, and they sit quietly inside almost every tokeniser, search box and data pipeline ever written. An hour spent here saves days later.

## 1. Describing text instead of spelling it out

### Intuition first
Suppose you must find every telephone number in a thousand documents. You cannot search for the numbers themselves, because you do not know them. What you do know is their **shape**: three digits, a dash, three digits, a dash, four digits.

A **regular expression** (or *regex*) is a small language for describing shapes of text. Instead of saying *"find 210-555-0199"*, you say *"find a digit, a digit, a digit, a dash..."* and the computer finds everything that fits.

In [ ]:
import re

text = "Call 210-555-0199 or 210-555-0142 before 17:30. Room 204."

print("210-555-0199" in text)                          # we can only test for text we already know
print(re.findall(r"\d\d\d-\d\d\d-\d\d\d\d", text))     # a pattern finds text we have never seen

### Formal definition
A regular expression is a string in which most characters stand for themselves and a few have a special meaning. In the pattern above, `\d` means *any one digit* and `-` means a literal dash.

Python's `re` module offers four functions that cover nearly every need:

| Function | What it returns |
|---|---|
| `re.search(pattern, text)` | the **first** match, or `None` |
| `re.findall(pattern, text)` | a **list of all** matches |
| `re.sub(pattern, replacement, text)` | the text with every match **replaced** |
| `re.split(pattern, text)` | the text **cut** at every match |

> 🧠 Always write patterns as **raw strings**, with an `r` in front: `r"\d"`. In an ordinary string Python treats the backslash as its own escape character and may change your pattern before the regex engine ever sees it.

## 2. Character classes: "one character from this set"

### Intuition first
A character class answers the question *"which characters are allowed in this position?"*. Square brackets list the allowed characters. A dash inside the brackets makes a range.

In [ ]:
text = "The cat sat on the mat with a bat, a rat and a hat."

print(re.findall(r"[cbr]at", text))      # c, b or r, then "at"
print(re.findall(r"[a-z]at", text))      # any lower-case letter, then "at"
print(re.findall(r"[^cbr ]at", text))    # ^ inside brackets means NOT: anything except c, b, r or a space

Some classes are needed so often that they have short names.

| Pattern | Matches |
|---|---|
| `\d` | one digit |
| `\w` | one "word character": a letter, a digit or an underscore |
| `\s` | one whitespace character: space, tab or new line |
| `\D`, `\W`, `\S` | the opposite of each of the above |
| `.` | any one character except a new line |

In [ ]:
text = "Order A17 cost 23.50 euros on 3 May."

print(re.findall(r"\d", text))       # single digits
print(re.findall(r"\w\d\d", text))   # a word character followed by two digits
print(re.findall(r"\d.\d", text))    # digit, anything, digit

The last pattern found `3.5`, which looks like what we wanted. Be careful, though: the dot in a pattern is not a full stop. It means *any* character, so the same pattern would also match `3x5` and `345`. If you want a real full stop, escape it with a backslash: `\.`

> ⚠️ **Common pitfalls**
>
> - Using `.` when you mean a literal full stop. `r"3.5"` also matches `"3x5"` and `"345"`. Write `r"3\.5"`.
> - The same is true for every special character: `. ^ $ * + ? ( ) [ ] { } | \`. To match one literally, put a backslash in front.
> - `\w` includes digits and the underscore. It is not "a letter".

## 3. Quantifiers: "how many times?"

### Intuition first
Writing `\d\d\d\d` for a four-digit year is tolerable. Writing a pattern for "a word of any length" that way is impossible. A **quantifier** follows a piece of pattern and says how many times it may repeat.

### Formal definition

| Quantifier | Meaning |
|---|---|
| `*` | zero or more times |
| `+` | one or more times |
| `?` | zero or one time (optional) |
| `{3}` | exactly 3 times |
| `{2,4}` | between 2 and 4 times |

In [ ]:
text = "In 1865 Carroll published Alice. By 2024 it had sold over 100 million copies in 174 languages."

print(re.findall(r"\d+", text))          # one or more digits: whole numbers
print(re.findall(r"\d{4}", text))        # exactly four digits: years
print(re.findall(r"[A-Z][a-z]+", text))  # a capital letter, then one or more small ones

In [ ]:
text = "colour color colouur"
print(re.findall(r"colou?r", text))      # the u is optional: British and American spelling

### Greedy and lazy
Quantifiers are **greedy**: they take as much text as they possibly can. That is often not what you want.

In [ ]:
html = "<b>bold</b> and <i>italic</i>"

print(re.findall(r"<.+>", html))     # greedy: from the first < to the LAST >
print(re.findall(r"<.+?>", html))    # lazy: from each < to the NEAREST >

Adding `?` after a quantifier makes it **lazy**: it takes as little as possible. The greedy pattern swallowed the whole line as a single match. The lazy one found the four tags separately.

A third option is usually the clearest of all: say exactly what you mean. A tag is a `<`, then characters that are *not* `>`, then a `>`.

In [ ]:
print(re.findall(r"<[^>]+>", html))

## 4. Anchors and boundaries: "where?"

### Intuition first
So far a pattern could match anywhere. **Anchors** do not match characters at all. They match **positions**.

| Anchor | Position |
|---|---|
| `^` | the start of the text |
| `$` | the end of the text |
| `\b` | a **word boundary**: the edge between a word character and a non-word character |

In [ ]:
text = "the cat scattered the catalogue; concatenate cat"

print(re.findall(r"cat", text))        # every "cat", even inside longer words
print(re.findall(r"\bcat\b", text))    # only the whole word "cat"
print(re.findall(r"\bcat\w*", text))   # words that START with "cat"

Word boundaries are what turn a substring search into a word search. Without them, a search for *"art"* returns *"party"*, *"smart"* and *"cartoon"*.

`^` and `$` are how you **validate** a whole string, as opposed to finding something inside it. `re.fullmatch` does the same job without the anchors.

In [ ]:
def looks_like_year(text):
    return re.fullmatch(r"(19|20)\d\d", text) is not None

for candidate in ["1987", "2024", "1787", "20245", "year 2024"]:
    print(candidate, looks_like_year(candidate))

Two new symbols appeared in that pattern. The vertical bar `|` means **or**: `19|20`. The round brackets form a **group**, which here limits how far the *or* reaches. Without the brackets, `19|20\d\d` would mean *"either 19, or 20 followed by two digits"*.

## 5. Groups: taking a match apart

### Intuition first
Finding a date is useful. Knowing which part is the day, which the month and which the year is more useful. Round brackets **capture** the part of the text they enclose, so that you can use it afterwards.

In [ ]:
text = "The course starts on 06/10/2025 and the exam is on 23/01/2026."

match = re.search(r"(\d{2})/(\d{2})/(\d{4})", text)

print(match.group(0))     # the whole match
print(match.group(1))     # the first group: the day
print(match.group(2))     # the second group: the month
print(match.group(3))     # the third group: the year

When a pattern has groups, `re.findall` returns the groups of every match, as tuples.

In [ ]:
for day, month, year in re.findall(r"(\d{2})/(\d{2})/(\d{4})", text):
    print(f"day {day}, month {month}, year {year}")

Groups can also be used inside a **replacement**. In `re.sub`, `\1`, `\2` and `\3` stand for whatever the first, second and third group captured. One line rewrites every date in the text into the international `year-month-day` format.

In [ ]:
print(re.sub(r"(\d{2})/(\d{2})/(\d{4})", r"\3-\2-\1", text))

> 🧠 If you need brackets only for grouping, and not for capturing, write `(?:...)`. For example `(?:19|20)\d\d` groups the two alternatives without creating a numbered group. This keeps `re.findall` returning whole matches.

## 6. Cleaning real text

Here is the kind of text that arrives from the web: a customer message with a link, an e-mail address, HTML, a price, capital letters and too many spaces.

In [ ]:
raw = """<p>GREAT service!!!   I ordered on 03/05/2024 &amp; it arrived in 2 days.</p>
Contact me at maria.p@example.com or see https://example.com/review?id=77 <br>
Total: 49.99 EUR.    Would   recommend :)"""

print(raw)

We clean it with a sequence of `re.sub` calls, each removing one kind of noise. Read the function one line at a time.

In [ ]:
def clean(text):
    text = text.lower()
    text = re.sub(r"https?://\S+", " ", text)      # web addresses
    text = re.sub(r"\S+@\S+", " ", text)           # e-mail addresses
    text = re.sub(r"<[^>]+>", " ", text)           # HTML tags
    text = re.sub(r"&\w+;", " ", text)             # HTML entities such as &amp;
    text = re.sub(r"[^a-z\s]", " ", text)          # everything that is not a letter or whitespace
    text = re.sub(r"\s+", " ", text)               # runs of whitespace become one space
    return text.strip()

print(clean(raw))

The **order of the steps matters**. Try to imagine the function with the "letters only" line moved to the top. It would delete the `:`, `/` and `@` characters first, and the later patterns for web and e-mail addresses would no longer recognise anything. The pieces `https`, `example` and `com` would then survive as if they were ordinary words.

> ⚠️ **Common pitfalls**
>
> - Cleaning in the wrong order. Remove the **structured** things (addresses, tags) first, while their punctuation is still there to identify them.
> - Cleaning too much. This function deletes all numbers and all punctuation. For a topic classifier that is fine. For a task that needs prices, dates or the difference between *"good."* and *"good?"*, it destroys the answer. Cleaning is a decision about what matters, never a neutral step.
> - Forgetting that `re.sub` returns a **new** string. Strings never change in place: you must assign the result.

## 7. A tokeniser in one line

In Notebook 01 we wrote `simple_words`, a loop of a dozen lines that kept letters and apostrophes. With a regular expression the same idea is a single pattern: *one or more letters, optionally followed by an apostrophe and more letters*.

In [ ]:
def regex_words(text):
    return re.findall(r"[a-z]+(?:'[a-z]+)*", text.lower())

print(regex_words("Don't panic: it's only 'text'!"))
print(regex_words("The cat sat on the mat. The dog did not."))

Instead of describing what to throw away, the pattern describes what a word **looks like**, and `findall` collects everything of that shape. Let us run it on a whole book and see how quickly it works.

In [ ]:
import nltk
nltk.download("gutenberg", quiet=True)
from nltk.corpus import gutenberg
from collections import Counter
import time

alice = gutenberg.raw("carroll-alice.txt")

start = time.perf_counter()
words = regex_words(alice)
elapsed = time.perf_counter() - start

print(f"{len(words):,} tokens and {len(set(words)):,} types in {elapsed:.3f} seconds")
print(Counter(words).most_common(8))

A tokeniser built on a pattern is easy to **extend**. Suppose we decide that numbers, including decimals such as `49.99`, should be tokens too. We add one alternative, and put it first so that it is tried first.

In [ ]:
def better_words(text):
    pattern = r"\d+(?:\.\d+)?|[a-z]+(?:'[a-z]+)*"
    return re.findall(pattern, text.lower())

print(better_words("It's 3.5 km to the station, or 12 minutes by bus."))

## 8. What regular expressions cannot do

Regular expressions describe the **shape** of text. They know nothing about meaning: the pattern `\d{2}/\d{2}/\d{4}` happily accepts the "date" `99/99/9999`.

There is also a deeper limit. A regular expression cannot **count to match things up**. Consider brackets that must be balanced, like `((()))`. No regular expression can accept every balanced string and reject every unbalanced one, however cleverly it is written.

In [ ]:
pattern = r"\(*\)*"      # some opening brackets, then some closing brackets

for candidate in ["()", "((()))", "(()", "())))"]:
    print(candidate, re.fullmatch(pattern, candidate) is not None)

The pattern accepts the two unbalanced strings as well, because it has no way to remember **how many** brackets it opened. This is not a weakness of Python's `re` module. It is a mathematical property of the whole family of patterns, and human language is full of structures of exactly this nested kind. Notebook 04 explains where the limit comes from, and Notebook 05 introduces the more powerful tool that goes beyond it.

---
## ✏️ Exercises

### Exercise 1 (Hashtags and mentions)
From the post below, extract a list of all hashtags (such as `#nlp`) and a separate list of all mentions (such as `@maria`).

In [ ]:
post = "Great talk by @maria and @j_smith on #nlp and #MachineLearning today! #AI2025"
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
post = "Great talk by @maria and @j_smith on #nlp and #MachineLearning today! #AI2025"

print(re.findall(r"#\w+", post))
print(re.findall(r"@\w+", post))
```

*Here it is useful that `\w` includes digits and the underscore.*
</details>

### Exercise 2 (Prices)
Extract every price from the text as a number. Prices may or may not have decimals, and the currency sign comes first. The result should be the list of floats `[4.5, 12.0, 0.99, 150.0]`.

In [ ]:
receipt = "Coffee €4.50, book €12, pen €0.99, and a jacket for €150."
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
receipt = "Coffee €4.50, book €12, pen €0.99, and a jacket for €150."

prices = []
for found in re.findall(r"€(\d+(?:\.\d+)?)", receipt):
    prices.append(float(found))
print(prices)
```

*The capturing group returns only the number, without the currency sign. The inner group is non-capturing so that `findall` still returns one string per price. Note that the full stop after 150 is not taken, because it is not followed by a digit.*
</details>

### Exercise 3 (Validate a postcode)
A Greek postcode is five digits, optionally written with a space after the third: `11524` or `115 24`. Write `is_postcode(text)` that returns `True` only for a correctly formed postcode. Test it on `"11524"`, `"115 24"`, `"1152"`, `"115  24"` and `"A1524"`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def is_postcode(text):
    return re.fullmatch(r"\d{3} ?\d{2}", text) is not None

for candidate in ["11524", "115 24", "1152", "115  24", "A1524"]:
    print(repr(candidate), is_postcode(candidate))
```

*`fullmatch` makes the pattern cover the whole string, so nothing extra is allowed before or after.*
</details>

### Exercise 4 (Words ending in -ing)
Using the text of *Alice in Wonderland* loaded above, find the ten most common words that end in `ing`. Count whole words only.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
ing_words = re.findall(r"\b[a-z]+ing\b", alice.lower())
print(Counter(ing_words).most_common(10))
```

*Half of the results are not verbs at all: "king", "thing", "nothing", "anything" and "something". A pattern sees the spelling of a word, not its grammar. Telling them apart is the subject of Notebook 06.*
</details>

### Exercise 5 (Hide personal data)
Write `anonymise(text)` that replaces every e-mail address with `[EMAIL]` and every telephone number of the form `210-555-0199` with `[PHONE]`.

In [ ]:
message = "Write to nikos@example.org or anna.k@uni.example.gr, or call 210-555-0199 / 231-555-0142."
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
message = "Write to nikos@example.org or anna.k@uni.example.gr, or call 210-555-0199 / 231-555-0142."

def anonymise(text):
    text = re.sub(r"[\w.]+@[\w.]+\w", "[EMAIL]", text)
    text = re.sub(r"\b\d{3}-\d{3}-\d{4}\b", "[PHONE]", text)
    return text

print(anonymise(message))
```

*The e-mail pattern ends with `\w` so that the comma after the address is not swallowed. Removing personal data before text is stored or shared is one of the most common real uses of regular expressions.*
</details>

### Exercise 6 (Split into sentences)
Use `re.split` to cut the text below into sentences. A sentence ends with `.`, `!` or `?` followed by whitespace. Then look at the result: where does this simple rule go wrong?

In [ ]:
paragraph = "Dr. Smith arrived at 9 a.m. on Monday. Was he late? Not at all! He paid 3.50 euros for coffee."
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
paragraph = "Dr. Smith arrived at 9 a.m. on Monday. Was he late? Not at all! He paid 3.50 euros for coffee."

for sentence in re.split(r"(?<=[.!?])\s+", paragraph):
    print(repr(sentence))
```

*`(?<=...)` is a "look-behind": it requires the punctuation to be there but does not remove it. The rule wrongly cuts after "Dr." and after "a.m.", because a full stop does not always end a sentence. It correctly leaves "3.50" alone, since no space follows that full stop. Notebook 03 shows how proper sentence splitters deal with abbreviations.*
</details>

### Exercise 7 (A tokeniser for the web, a little harder)
Extend `better_words` into `web_words(text)` so that it also keeps as single tokens: hashtags (`#nlp`), mentions (`@maria`) and hyphenated words (`state-of-the-art`). Keep numbers and ordinary words working as before.

In [ ]:
sample = "@maria says the state-of-the-art model costs 3.5 times less. It's #amazing, isn't it?"
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
sample = "@maria says the state-of-the-art model costs 3.5 times less. It's #amazing, isn't it?"

def web_words(text):
    pattern = r"[#@]\w+|\d+(?:\.\d+)?|[a-z]+(?:['-][a-z]+)*"
    return re.findall(pattern, text.lower())

print(web_words(sample))
```

*The alternatives are tried from left to right, so the most specific shapes come first. Allowing either an apostrophe or a hyphen inside a word handles "isn't" and "state-of-the-art" with the same piece of pattern.*
</details>

## 🔑 Key takeaways

- A **regular expression** describes the shape of the text you are looking for, so it can find text you have never seen.
- **Character classes** (`[a-z]`, `\d`, `\w`, `\s`) say *which* characters, **quantifiers** (`*`, `+`, `?`, `{m,n}`) say *how many*, and **anchors** (`^`, `$`, `\b`) say *where*.
- Quantifiers are **greedy** by default. Add `?` to make them lazy, or better, use a negated class such as `[^>]+`.
- **Groups** capture parts of a match, for extraction with `findall` and for reuse in `re.sub`.
- A cleaning pipeline is a sequence of substitutions, and the **order matters**: structured items first.
- Cleaning always throws information away. Decide what your task needs before you delete it.
- Regular expressions see spelling, not meaning, and they **cannot handle nested structure**.

---
**Next:** *Notebook 03: Tokenisation and Normalisation*, where we move from our own patterns to professional tokenisers, and decide which words to keep, which to drop, and which to treat as the same.

---
*© Ioannis Tsioulis. *From Words to Meaning: Natural Language Processing in Python*. Prepared for educational use.*